In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd "drive/MyDrive/reasoning"

/content/drive/MyDrive/reasoning


In [3]:
from helpers.qwen3 import download_qwen3_small

download_qwen3_small(kind="base", tokenizer_only=True, out_dir="qwen3") 

In [4]:
from pathlib import Path
from helpers.qwen3 import Qwen3Tokenizer

tokenizer_path = Path("qwen3") / "tokenizer-base.json"
tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

In [5]:
prompt = "Explain large language models."
input_token_ids_to_list = tokenizer.encode(prompt)

In [6]:
for i in input_token_ids_to_list:
    print(f"{i} --> {tokenizer.decode([i])}")

840 --> Ex
20772 --> plain
3460 -->  large
4128 -->  language
4119 -->  models
13 --> .


In [7]:
text = tokenizer.decode(input_token_ids_to_list)
print(text)

Explain large language models.


In [8]:
import torch
device = torch.device("cpu")

In [9]:
download_qwen3_small(kind="base", tokenizer_only=False, out_dir="qwen3") 

✓ qwen3/qwen3-0.6B-base.pth already up-to-date


In [10]:
from helpers.qwen3 import Qwen3Model, QWEN_CONFIG_06_B

model_path = Path("qwen3") / "qwen3-0.6B-base.pth"

model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(model_path))

model.to(device)

Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

In [12]:
prompt = "Explain large language models."
input_token_ids_to_list = tokenizer.encode(prompt)
print(f"Number of input tokens: {len(input_token_ids_to_list)}")

input_tensor = torch.tensor(input_token_ids_to_list)
input_tensor_fmt = input_tensor.unsqueeze(0).to(device)

with torch.inference_mode():
    output_tensor = model(input_tensor_fmt)

output_tensor_fmt = output_tensor.squeeze(0)
print(f"Formatted Output tensor shape: {output_tensor_fmt.shape}")

Number of input tokens: 6
Formatted Output tensor shape: torch.Size([6, 151936])


In [13]:
last_token = output_tensor_fmt[-1]
print(last_token)

tensor([ 7.3125,  1.9453,  7.9062,  ..., -2.4688, -2.4688, -2.4688],
       dtype=torch.bfloat16)


In [14]:
print(torch.argmax(last_token, dim=-1, keepdim=True))

tensor([20286])


In [15]:
print(tokenizer.decode([20286]))

 Large


In [ ]:
@torch.inference_mode()
def generate_text_basic_stream(model, token_ids, max_new_tokens, eos_token_id=None):
    model.eval()
    
    for _ in range(max_new_tokens):
        out = model(token_ids)[:, -1]
        next_token = torch.argmax(out, dim=-1, keepdim=True)
        
        if (eos_token_id is not None and torch.all(next_token == eos_token_id)):
            break
        
        yield next_token
        
        token_ids = torch.cat([token_ids, next_token], dim=1)

In [25]:
prompt = "Explain Large Language Models in a single sentence."
input_token_ids_tensor = torch.tensor(tokenizer.encode(prompt), device=device).unsqueeze(0)
max_new_tokens = 100

for token in generate_text_basic_stream(model=model, token_ids=input_token_ids_tensor, max_new_tokens=max_new_tokens,):
    token_id = token.squeeze(0).tolist()
    print(tokenizer.decode(token_id), end="", flush=True)

 Large Language Models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform tasks such as translation, summarization, and question answering.<|endoftext|>Human language is a complex and dynamic system that requires sophisticated models to understand and generate meaningful responses. Large Language Models are designed to mimic this complexity by learning from vast amounts of text data and using advanced algorithms to process and interpret language. These models can be trained on a wide range of sources, including books, articles, and social

In [18]:
print(tokenizer.encode("<|endoftext|>"))

[151643]


In [19]:
print(tokenizer.eos_token_id)

151643


In [21]:
prompt = "Explain Large Language Models in a single sentence."
input_token_ids_tensor = torch.tensor(tokenizer.encode(prompt), device=device).unsqueeze(0)
max_new_tokens = 100

for token in generate_text_basic_stream(model=model, token_ids=input_token_ids_tensor, max_new_tokens=max_new_tokens, eos_token_id=tokenizer.eos_token_id):
    token_id = token.squeeze(0).tolist()
    print(tokenizer.decode(token_id), end="", flush=True)

 Large Language Models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform tasks such as translation, summarization, and question answering.

In [23]:
from helpers.qwen3 import KVCache

@torch.inference_mode()
def generate_text_basic_stream(model, token_ids, max_new_tokens, eos_token_id=None):
    model.eval()
    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()
    
    out = model(token_ids, cache=cache)[:, -1]    
    for step in range(max_new_tokens):
        next_token = torch.argmax(out, dim=-1, keepdim=True)
        
        if (eos_token_id is not None and torch.all(next_token == eos_token_id)):
            break
        
        yield next_token
        if step + 1 < max_new_tokens:
            out = model(next_token, cache=cache)[:, -1]

In [24]:
generated_ids = []
for token in generate_text_basic_stream(model=model, token_ids=input_token_ids_tensor, max_new_tokens=max_new_tokens, eos_token_id=tokenizer.eos_token_id):
    token_id = token.squeeze(0).tolist()
    print(tokenizer.decode(token_id), end="", flush=True)
    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)
    
output_token_ids_tensor = torch.cat(generated_ids, dim=0)

 Large Language Models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform tasks such as translation, summarization, and question answering.